In [ ]:
!git clone https://github.com/williamqwu/ml-tutorials-suite.git
%cd ml-tutorials-suite

Cloning into 'ml-tutorials-suite'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 152 (delta 69), reused 144 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 4.50 MiB | 13.12 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/ml-tutorials-suite


In [ ]:
# Week 4 Assignment

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from model.vitfc import ViTForClassfication

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

batch_size = 256
learning_rate = 1e-3
num_epochs = 20

vit_config = {
    "patch_size": 4,
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 4 * 48,
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 28,
    "num_classes": 10,
    "num_channels": 1,
    "qkv_bias": True,
    "use_faster_attention": True,
}

# Load data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_data = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_data = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_data, batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_data, batch_size=batch_size, shuffle=False
)

# Set up model
model = ViTForClassfication(vit_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

# Training loop
for epoch in range(num_epochs):

    # train
    model.train()
    train_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(images)[0]
        loss = loss_fn(predictions, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_loader)

    # evaluate
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images)[0]
            test_loss += loss_fn(predictions, labels).item()
            correct += (predictions.argmax(dim=1) == labels).sum().item()

    test_loss /= len(test_loader)
    accuracy = correct / len(test_data)

    print(
        f"Epoch {epoch + 1}: train loss {train_loss:.4f}, "
        f"test loss {test_loss:.4f}, test accuracy {accuracy * 100:.2f}%"
    )

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")
print("Done training!")

Device: cpu


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 16.9MB/s]


Epoch 1: train loss 1.4715, test loss 0.9655, test accuracy 61.48%
Epoch 2: train loss 0.8239, test loss 0.6953, test accuracy 73.24%
Epoch 3: train loss 0.6518, test loss 0.6126, test accuracy 77.71%
Epoch 4: train loss 0.5605, test loss 0.5696, test accuracy 78.65%
Epoch 5: train loss 0.5035, test loss 0.5149, test accuracy 80.66%
Epoch 6: train loss 0.4777, test loss 0.4914, test accuracy 81.38%
Epoch 7: train loss 0.4486, test loss 0.4670, test accuracy 82.84%
Epoch 8: train loss 0.4284, test loss 0.4403, test accuracy 83.58%
Epoch 9: train loss 0.4154, test loss 0.4485, test accuracy 83.78%
Epoch 10: train loss 0.4059, test loss 0.4158, test accuracy 84.44%
Epoch 11: train loss 0.3986, test loss 0.4286, test accuracy 84.08%
Epoch 12: train loss 0.3886, test loss 0.3995, test accuracy 85.12%
Epoch 13: train loss 0.3871, test loss 0.4071, test accuracy 84.49%
Epoch 14: train loss 0.3730, test loss 0.4272, test accuracy 84.33%
Epoch 15: train loss 0.3719, test loss 0.4070, test accur